# 🚀 Space Missions & Launches — Complete EDA (1957–2024)
## Sputnik → Apollo → Shuttle → SpaceX · 67 Years of Space History

**Dataset:** Space Missions & Launches (1957–2024)  
**Author:** Sergey Nefedov | [github.com/Sergpreneur](https://github.com/Sergpreneur)

---

### What this notebook covers
1. 📊 Overview — 67 years of launches, eras, success rates
2. 🌍 Space race — USSR vs USA Cold War competition
3. 🔄 The SpaceX disruption — cost revolution & reusability
4. 🛰️ Payload & orbit analysis — what goes where and why
5. 🏆 Country comparison — rise of China, commercial players
6. 🚀 Rocket analysis — which vehicles defined each era
7. 🤖 Launch success prediction — ML model

> **Key numbers:**  
> Sputnik 1957 → 247 launches in 2023 alone  
> Cost to LEO: $65,000/kg (1970) → $2,700/kg (Falcon 9, 2024) — a **24× reduction**  
> SpaceX Falcon 9 success rate: **99%** — highest of any orbital rocket in history


## 0. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130, 'axes.facecolor': '#050812', 'figure.facecolor': '#050812',
    'axes.edgecolor': '#1e2a4a', 'axes.labelcolor': '#c9d1d9',
    'xtick.color': '#8b949e', 'ytick.color': '#8b949e', 'text.color': '#c9d1d9',
    'grid.color': '#0d1520', 'grid.alpha': 0.6,
    'axes.spines.top': False, 'axes.spines.right': False,
})
BLUE='#388bfd'; GREEN='#3fb950'; RED='#f85149'; AMBER='#f7931a'
PURPLE='#9945ff'; TEAL='#39d353'; GRAY='#8b949e'; GOLD='#e8a020'

ERA_COLORS = {
    'Cold War':RED,'Post-Cold War':AMBER,'Post-Soviet':GOLD,
    'Modern':BLUE,'New Space':GREEN
}
COUNTRY_COLORS = {
    'USSR':RED,'Russia':AMBER,'USA':BLUE,'China':RED,
    'Europe':GREEN,'Japan':PURPLE,'India':TEAL,'Other':GRAY
}

import os, glob
def find_path():
    candidates = [
        '/kaggle/input/space-missions-launches-1957-2024/',
        '/kaggle/input/space-missions-1957-2024/',
        '/kaggle/input/space-launches/',
    ]
    for c in candidates:
        if os.path.exists(c + 'launches.csv'):
            return c
    matches = glob.glob('/kaggle/input/**/launches.csv', recursive=True)
    if matches: return os.path.dirname(matches[0]) + '/'
    return './'

PATH = find_path()

launches = pd.read_csv(PATH + 'launches.csv', parse_dates=['date'])
orgs     = pd.read_csv(PATH + 'organizations.csv')
rockets  = pd.read_csv(PATH + 'rockets.csv')
payloads = pd.read_csv(PATH + 'payloads.csv')
sites    = pd.read_csv(PATH + 'launch_sites.csv')

launches['year']  = launches['date'].dt.year
launches['decade'] = (launches['year']//10)*10

print(f"Launches:  {len(launches):>7,} | {launches['year'].min()}–{launches['year'].max()}")
print(f"Rockets:   {len(rockets):>7,} | {rockets['country'].nunique()} countries")
print(f"Orgs:      {len(orgs):>7,}")
print(f"Payloads:  {len(payloads):>7,}")
print(f"Sites:     {len(sites):>7,}")
print(f"\nSuccess rate: {launches['launch_success'].mean():.1%}")
print(f"Crewed missions: {launches['crewed'].sum():,}")
print(f"Reused boosters: {launches['booster_reused'].sum():,}")
print(f"\nLaunches 2023: {(launches.year==2023).sum():,}")
print(f"\nTop operators:")
print(launches['operator'].value_counts().head(8).to_string())


---
## 1. 📊 Overview — 67 Years of Space Launches

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Panel 1: Annual launches by era
ax = axes[0,0]
for era, color in ERA_COLORS.items():
    sub = launches[launches['rocket_era']==era]
    annual_era = sub.groupby('year').size()
    ax.fill_between(annual_era.index, annual_era.values, alpha=0.3, color=color)
    ax.plot(annual_era.index, annual_era.values, color=color, linewidth=1.5, label=era)
ax.set_title('Annual Launches by Era (1957–2024)', fontsize=11)
ax.set_ylabel('Launches per year'); ax.legend(fontsize=7, ncol=2); ax.grid(True, alpha=0.3)

# Panel 2: Success rate over time
ax = axes[0,1]
success_annual = launches.groupby('year')['launch_success'].mean() * 100
roll = success_annual.rolling(5, center=True).mean()
ax.fill_between(success_annual.index, success_annual.values, alpha=0.2, color=GREEN)
ax.plot(success_annual.index, success_annual.values, color=GREEN, linewidth=1, alpha=0.5)
ax.plot(roll.index, roll.values, color=GREEN, linewidth=2.5, label='5-yr rolling avg')
ax.axhline(99, color=BLUE, linewidth=1, linestyle='--', label='99% (Falcon 9 FT level)')
ax.set_title('Launch Success Rate Over Time (%)', fontsize=11)
ax.set_ylabel('%'); ax.set_ylim(40, 102)
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Panel 3: Mission type distribution
ax = axes[0,2]
mt_counts = launches['mission_type'].value_counts().head(10)
ax.barh(range(len(mt_counts)), mt_counts.values,
        color=[RED,BLUE,GREEN,AMBER,PURPLE,TEAL,GOLD,GRAY,RED,BLUE], alpha=0.85)
ax.set_yticks(range(len(mt_counts)))
ax.set_yticklabels([t[:28] for t in mt_counts.index], fontsize=8)
ax.set_title('Top 10 Mission Types', fontsize=11)
ax.set_xlabel('Count'); ax.grid(True, alpha=0.3, axis='x')

# Panel 4: Orbit distribution
ax = axes[1,0]
orbit_counts = launches['orbit'].value_counts()
colors_pie = [BLUE,GREEN,AMBER,RED,PURPLE,TEAL,GOLD,GRAY,BLUE,GREEN]
wedges,texts,autotexts = ax.pie(
    orbit_counts.values[:8], labels=orbit_counts.index[:8],
    colors=colors_pie, autopct='%1.0f%%', startangle=90,
    wedgeprops=dict(edgecolor='#050812',linewidth=1.5),
    textprops={'fontsize':8,'color':'#c9d1d9'})
for at in autotexts: at.set_color('#c9d1d9'); at.set_fontsize(8)
ax.set_title('Launch Orbit Distribution', fontsize=11)

# Panel 5: Crewed missions per year
ax = axes[1,1]
crewed_annual = launches[launches['crewed']==1].groupby('year').size()
ax.fill_between(crewed_annual.index, crewed_annual.values, alpha=0.4, color=AMBER)
ax.plot(crewed_annual.index, crewed_annual.values, color=AMBER, linewidth=2)
ax.axvspan(1972, 1981, alpha=0.1, color=RED, label='Gap (Apollo end)')
ax.axvspan(2011, 2020, alpha=0.1, color=RED, label='US crewed gap')
ax.set_title('Crewed Launches per Year', fontsize=11)
ax.set_ylabel('Crewed Launches'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Panel 6: Failure modes
ax = axes[1,2]
failures = launches[launches['launch_success']==0]['failure_mode'].value_counts()
ax.barh(failures.index, failures.values, color=RED, alpha=0.85)
ax.set_title('Launch Failure Causes', fontsize=11)
ax.set_xlabel('Count'); ax.grid(True, alpha=0.3, axis='x')

plt.suptitle('Global Space Launch Overview 1957–2024', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('overview.png', dpi=130, bbox_inches='tight', facecolor='#050812')
plt.show()

print(f"Overall success rate: {launches['launch_success'].mean():.1%}")
print(f"New Space era (2016+): {launches[launches.year>=2016]['launch_success'].mean():.2%}")


---
## 2. 🌍 The Space Race — USSR vs USA Cold War

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

cold_war = launches[launches['year'] <= 1991].copy()
cold_war['bloc'] = cold_war['country'].map(
    lambda x: 'USSR/Russia' if x in ['USSR','Russia'] else
              'USA' if x == 'USA' else 'Other')

# Panel 1: Annual launches USSR vs USA
ax = axes[0,0]
for bloc, color, lw in [('USSR/Russia',RED,2.5),('USA',BLUE,2.5),('Other',GRAY,1.5)]:
    sub = cold_war[cold_war['bloc']==bloc].groupby('year').size()
    ax.fill_between(sub.index, sub.values, alpha=0.2, color=color)
    ax.plot(sub.index, sub.values, color=color, linewidth=lw, label=bloc)
# Annotate key moments
for yr, label, y in [(1957,'Sputnik',5),(1961,'Gagarin',8),(1969,'Apollo 11',35),
                     (1971,'Salyut',10),(1981,'Shuttle',8),(1986,'Challenger',6)]:
    ax.axvline(yr, color=GRAY, linewidth=0.7, linestyle=':', alpha=0.6)
    ax.text(yr+0.3, y, label, fontsize=7, color='#c9d1d9', rotation=90)
ax.set_title('Cold War Space Race: Annual Launches (1957–1991)', fontsize=11)
ax.set_ylabel('Launches per year'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Panel 2: Cumulative launches
ax = axes[0,1]
for bloc, color in [('USSR/Russia',RED),('USA',BLUE)]:
    sub = cold_war[cold_war['bloc']==bloc].groupby('year').size().cumsum()
    ax.plot(sub.index, sub.values, color=color, linewidth=2.5, label=bloc)
ax.set_title('Cumulative Launches: USSR vs USA Cold War', fontsize=11)
ax.set_ylabel('Cumulative Launches')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Panel 3: Success rate comparison
ax = axes[1,0]
for bloc, color in [('USSR/Russia',RED),('USA',BLUE)]:
    sub = cold_war[cold_war['bloc']==bloc].groupby('year')['launch_success'].mean()*100
    roll = sub.rolling(5,center=True).mean()
    ax.plot(sub.index, roll.values, color=color, linewidth=2.5, label=bloc)
ax.set_title('Success Rate: USSR vs USA (5-yr rolling, %)', fontsize=11)
ax.set_ylabel('%'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Panel 4: Mission type comparison
ax = axes[1,1]
for bloc, color, offset in [('USSR/Russia',RED,-0.2),('USA',BLUE,0.2)]:
    sub = cold_war[cold_war['bloc']==bloc]
    mt_counts = sub['mission_type'].value_counts().head(6)
    x_ = np.arange(len(mt_counts))
    ax.bar(x_+offset, mt_counts.values, 0.38, color=color, alpha=0.85, label=bloc)
    ax.set_xticks(x_)
    ax.set_xticklabels([t[:18] for t in mt_counts.index], rotation=30, ha='right', fontsize=7)
ax.set_title('Mission Types: USSR vs USA Cold War', fontsize=11)
ax.set_ylabel('Count'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('The Cold War Space Race (1957–1991)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('space_race.png', dpi=130, bbox_inches='tight', facecolor='#050812')
plt.show()

ussr_total = len(cold_war[cold_war['bloc']=='USSR/Russia'])
usa_total  = len(cold_war[cold_war['bloc']=='USA'])
print(f"USSR/Russia Cold War launches: {ussr_total:,}")
print(f"USA Cold War launches:         {usa_total:,}")
print(f"Soviet dominance ratio: {ussr_total/usa_total:.2f}× more launches")


---
## 3. 🔄 The SpaceX Disruption — Cost Revolution & Reusability

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Panel 1: Cost per kg over time
ax = axes[0,0]
cost_rockets = rockets[rockets['cost_per_kg_usd'].notna()].sort_values('first_flight')
ax.scatter(cost_rockets['first_flight'], cost_rockets['cost_per_kg_usd'],
           c=[GREEN if r else RED for r in cost_rockets['reusable']],
           s=80, alpha=0.85, zorder=5)
# Trend line (expendable only)
expend = cost_rockets[cost_rockets['reusable']==0]
if len(expend) > 3:
    z = np.polyfit(expend['first_flight'], np.log(expend['cost_per_kg_usd'].clip(1)), 1)
    x_line = np.linspace(expend['first_flight'].min(), expend['first_flight'].max(), 100)
    ax.plot(x_line, np.exp(np.polyval(z,x_line)), color=RED, linewidth=1.5,
            linestyle='--', alpha=0.7, label='Expendable trend')
ax.set_yscale('log')
ax.set_title('Cost per kg to LEO Over Time (log)', fontsize=11)
ax.set_ylabel('USD/kg (log scale)')
red_p  = mpatches.Patch(color=RED,   label='Expendable')
green_p= mpatches.Patch(color=GREEN, label='Reusable')
ax.legend(handles=[red_p,green_p], fontsize=9); ax.grid(True, alpha=0.3)
# Annotate Falcon 9
f9 = cost_rockets[cost_rockets['rocket']=='Falcon 9 FT']
if not f9.empty:
    ax.annotate('Falcon 9 FT',
                xy=(f9['first_flight'].iloc[0], f9['cost_per_kg_usd'].iloc[0]),
                xytext=(20,20), textcoords='offset points',
                fontsize=8, color=GREEN,
                arrowprops=dict(arrowstyle='->', color=GREEN, lw=1))

# Panel 2: SpaceX launches per year
ax = axes[0,1]
spacex = launches[launches['operator']=='SpaceX']
non_spacex = launches[launches['operator']!='SpaceX']
spacex_annual = spacex.groupby('year').size()
non_sx_annual = non_spacex[non_spacex['year']>=2010].groupby('year').size()
ax2 = ax.twinx()
ax.bar(spacex_annual.index, spacex_annual.values, color=GREEN, alpha=0.85, label='SpaceX')
ax2.plot(non_sx_annual.index, non_sx_annual.values, color=BLUE, linewidth=2,
         marker='o', markersize=5, label='All others')
ax.set_title('SpaceX vs World Annual Launches (2010+)', fontsize=11)
ax.set_ylabel('SpaceX launches', color=GREEN)
ax2.set_ylabel('All other launches', color=BLUE)
lines1,labels1=ax.get_legend_handles_labels()
lines2,labels2=ax2.get_legend_handles_labels()
ax.legend(lines1+lines2,labels1+labels2,fontsize=8); ax.grid(True,alpha=0.3,axis='y')

# Panel 3: Booster reuse over time
ax = axes[1,0]
reuse_annual = launches[launches['booster_reused']==1].groupby('year').size()
total_falcon  = launches[launches['rocket'].str.contains('Falcon',na=False)].groupby('year').size()
ax.fill_between(reuse_annual.index, reuse_annual.values, alpha=0.4, color=GREEN)
ax.plot(reuse_annual.index, reuse_annual.values, color=GREEN, linewidth=2.5, label='Reused boosters')
ax.plot(total_falcon.index, total_falcon.values, color=BLUE, linewidth=1.5,
        linestyle='--', label='Total Falcon launches')
ax.axvline(2017, color=AMBER, linewidth=1.5, linestyle='--', label='First reuse (2017)')
ax.set_title('Falcon Booster Reuse Program', fontsize=11)
ax.set_ylabel('Launches'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

# Panel 4: Payload mass capacity evolution
ax = axes[1,1]
for era, color in ERA_COLORS.items():
    sub = rockets[rockets['era']==era]
    if len(sub) > 2:
        ax.scatter([era]*len(sub), sub['payload_to_leo_kg'],
                   color=color, alpha=0.6, s=60, label=era)
ax.set_title('Payload to LEO by Era (kg)', fontsize=11)
ax.set_ylabel('Max LEO payload (kg)')
ax.set_yscale('symlog', linthresh=1000)
ax.legend(fontsize=7); ax.grid(True, alpha=0.3, axis='y')
# Annotate Saturn V and Starship
for rname, label in [('Saturn V','Saturn V'),('Starship (IFT)','Starship')]:
    sub = rockets[rockets['rocket']==rname]
    if not sub.empty:
        era = sub['era'].iloc[0]
        ax.annotate(label, xy=(era, sub['payload_to_leo_kg'].iloc[0]),
                    xytext=(5,5), textcoords='offset points', fontsize=7, color='#c9d1d9')

plt.suptitle('The SpaceX Revolution: Cost, Reusability & Market Share', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('spacex.png', dpi=130, bbox_inches='tight', facecolor='#050812')
plt.show()

spacex_share = len(spacex[spacex['year']>=2020]) / len(launches[launches['year']>=2020])
print(f"SpaceX share of US launches 2020+: {spacex_share:.1%}")


---
## 4. 🌍 Country Competition & 🚀 Rocket Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Panel 1: Country shares over time (stacked area)
ax = axes[0,0]
country_annual = launches.groupby(['year','country']).size().unstack(fill_value=0)
key_countries = ['USSR','Russia','USA','China','Europe','Japan','India','Other']
key_countries = [c for c in key_countries if c in country_annual.columns]
colors_c = [RED,AMBER,BLUE,RED,GREEN,PURPLE,TEAL,GRAY][:len(key_countries)]
country_annual[key_countries].plot.area(ax=ax, color=colors_c, alpha=0.8, linewidth=0)
ax.set_title('Annual Launches by Country (stacked)', fontsize=11)
ax.set_ylabel('Launches'); ax.legend(fontsize=7, ncol=2, loc='upper left')
ax.grid(True, alpha=0.3)

# Panel 2: China's rise
ax = axes[0,1]
china = launches[launches['country']=='China'].groupby('year').size()
usa   = launches[launches['country']=='USA'].groupby('year').size()
russia= launches[launches['country'].isin(['Russia','USSR'])].groupby('year').size()
for series, color, label in [(china,RED,'China'),(usa,BLUE,'USA'),(russia,AMBER,'Russia/USSR')]:
    ax.plot(series.index, series.values, color=color, linewidth=2.5, label=label)
ax.set_title('Launch Leaders: USA vs China vs Russia/USSR', fontsize=11)
ax.set_ylabel('Launches per year'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax.set_xlim(2000, 2024)

# Panel 3: Top rockets by total launches
ax = axes[1,0]
top_rockets = launches['rocket'].value_counts().head(12).sort_values(ascending=True)
ax.barh(range(len(top_rockets)), top_rockets.values,
        color=[GREEN if 'Falcon' in r else RED if any(x in r for x in ['Soyuz','Proton','Cosmos'])
               else BLUE for r in top_rockets.index], alpha=0.85)
ax.set_yticks(range(len(top_rockets)))
ax.set_yticklabels(top_rockets.index, fontsize=8)
ax.set_title('Most Frequently Launched Rockets', fontsize=11)
ax.set_xlabel('Total Launches'); ax.grid(True, alpha=0.3, axis='x')

# Panel 4: Rocket success rate vs launches (bubble chart)
ax = axes[1,1]
rocket_stats = launches.groupby('rocket').agg(
    n_launches=('launch_success','count'),
    success_rate=('launch_success','mean')
).reset_index()
rocket_stats = rocket_stats[rocket_stats['n_launches'] >= 10]
sc = ax.scatter(rocket_stats['n_launches'], rocket_stats['success_rate']*100,
                s=rocket_stats['n_launches']*0.5, alpha=0.6,
                c=rocket_stats['success_rate'], cmap='RdYlGn', vmin=0.5, vmax=1.0)
plt.colorbar(sc, ax=ax, label='Success rate')
ax.axhline(95, color=GRAY, linewidth=0.8, linestyle='--')
ax.set_title('Rockets: Launch Count vs Success Rate (%)', fontsize=11)
ax.set_xlabel('Total Launches'); ax.set_ylabel('Success Rate (%)')
ax.grid(True, alpha=0.3)
# Annotate notable rockets
for rname in ['Falcon 9 FT','Soyuz-U','Long March 3B','Proton-M']:
    sub = rocket_stats[rocket_stats['rocket']==rname]
    if not sub.empty:
        ax.annotate(rname[:12], (sub['n_launches'].iloc[0], sub['success_rate'].iloc[0]*100),
                    fontsize=7, color='#c9d1d9', xytext=(4,4), textcoords='offset points')

plt.suptitle('Country Competition & Rocket Performance', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('countries_rockets.png', dpi=130, bbox_inches='tight', facecolor='#050812')
plt.show()

china_2023 = (launches[(launches.country=='China')&(launches.year==2023)]).shape[0]
usa_2023   = (launches[(launches.country=='USA')&(launches.year==2023)]).shape[0]
print(f"China launches 2023: {china_2023} | USA: {usa_2023}")


---
## 5. 🤖 Launch Success Prediction

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.metrics import roc_curve, precision_recall_curve
import warnings; warnings.filterwarnings('ignore')

ml = launches.copy()
for col in ['rocket','country','mission_type','orbit','rocket_era','rocket_type']:
    ml[col+'_enc'] = LabelEncoder().fit_transform(ml[col].fillna('Unknown'))

FEATURES = ['year','month','rocket_enc','country_enc','mission_type_enc',
            'orbit_enc','rocket_era_enc','rocket_type_enc','crewed']

ml = ml.dropna(subset=FEATURES+['launch_success'])
X = ml[FEATURES]; y = ml['launch_success']

# Time split: train pre-2010, test 2010+
train_mask = ml['year'] < 2010
X_tr, X_te = X[train_mask], X[~train_mask]
y_tr, y_te = y[train_mask], y[~train_mask]

gbm = GradientBoostingClassifier(n_estimators=150, max_depth=4,
                                  learning_rate=0.05, random_state=42)
lr  = Pipeline([('sc',RobustScaler()),
                ('clf',LogisticRegression(max_iter=500,random_state=42))])

gbm.fit(X_tr,y_tr); lr.fit(X_tr,y_tr)
gbm_proba = gbm.predict_proba(X_te)[:,1]
lr_proba  = lr.predict_proba(X_te)[:,1]
gbm_auc   = roc_auc_score(y_te, gbm_proba)
gbm_ap    = average_precision_score(y_te, gbm_proba)
lr_auc    = roc_auc_score(y_te, lr_proba)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Feature importance
ax = axes[0]
fi = pd.Series(gbm.feature_importances_, index=FEATURES).sort_values(ascending=True)
ax.barh(fi.index, fi.values,
        color=[RED if 'rocket' in f else AMBER if 'era' in f else
               GREEN if 'year' in f else BLUE for f in fi.index],
        alpha=0.85)
ax.set_title(f'Feature Importance (GBM AUC={gbm_auc:.3f})', fontsize=11)

ax.set_xlabel('Importance'); ax.grid(True, alpha=0.3, axis='x')

# ROC curve
ax = axes[1]
fpr_g,tpr_g,_ = roc_curve(y_te, gbm_proba)
fpr_l,tpr_l,_ = roc_curve(y_te, lr_proba)
ax.plot(fpr_g, tpr_g, color=BLUE, linewidth=2.5, label=f'GBM (AUC={gbm_auc:.3f})')
ax.plot(fpr_l, tpr_l, color=AMBER, linewidth=2, linestyle='--', label=f'Logistic (AUC={lr_auc:.3f})')
ax.plot([0,1],[0,1], color=GRAY, linewidth=0.8, linestyle=':')
ax.set_title('ROC Curve — Launch Success Prediction', fontsize=11)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Success rate prediction by year (test set)
ax = axes[2]
test_df = ml[~train_mask].copy()
test_df['predicted_prob'] = gbm_proba
yearly_actual = test_df.groupby('year')['launch_success'].mean()*100
yearly_pred   = test_df.groupby('year')['predicted_prob'].mean()*100
ax.plot(yearly_actual.index, yearly_actual.values, color=GREEN, linewidth=2, label='Actual')
ax.plot(yearly_pred.index, yearly_pred.values, color=BLUE, linewidth=2,
        linestyle='--', label='Predicted')
ax.fill_between(yearly_actual.index, yearly_actual.values, yearly_pred.values, alpha=0.15, color=RED)
ax.set_title('Predicted vs Actual Success Rate by Year', fontsize=11)
ax.set_ylabel('Success Rate (%)'); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.suptitle(f'Launch Success Prediction | GBM AUC={gbm_auc:.3f} | PR-AUC={gbm_ap:.3f}',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('model.png', dpi=130, bbox_inches='tight', facecolor='#050812')
plt.show()

print(f"GBM: ROC-AUC={gbm_auc:.4f} | PR-AUC={gbm_ap:.4f}")
print(f"Logistic: ROC-AUC={lr_auc:.4f}")
print(f"Top feature: {fi.idxmax()}")
print(f"\nNote: High AUC reflects that rocket identity alone predicts success well")
print(f"(New Space rockets have near-perfect records; Soviet-era had ~85-90%)")


---
## 6. 📋 Key Findings

### The Space Race (1957–1991)
The USSR launched **{ussr}× more missions** than the USA during the Cold War — the Soviet space program was primarily military and automated (Cosmos series), while the US program had more prestige human missions (Apollo). Both reached success rates above 95% by the late 1970s.

### The Commercial Revolution
SpaceX's Falcon 9 achieved three historically unprecedented things simultaneously:
1. **Highest reliability** of any orbital rocket (99%+ success rate)
2. **Lowest cost** to LEO (~$2,700/kg vs ~$10,000/kg Atlas V)
3. **Reusability at scale** — the same boosters flew 10+ times

### China's Rise
China surpassed Russia in annual launch count around 2018 and is on track to challenge US dominance by 2026-27. The Long March family has achieved 95%+ success rates — comparable to European Ariane 5.

### The Payload Revolution
The rideshare model (n_payloads > 1) emerged post-2018. SpaceX's Transporter missions carry 80-143 satellites per launch. This democratized space access: a 3U cubesat can now reach SSO for ~$5,000.

### ML Prediction
Rocket identity is the dominant feature — each vehicle has a well-characterized historical success rate. Year adds marginal information beyond rocket type. The model correctly identifies that "which rocket" predicts success far better than "what mission."

---

*Dataset & notebook by **Sergey Nefedov** | [github.com/Sergpreneur](https://github.com/Sergpreneur)*  
*Sources: Jonathan's Space Report, NASA NSSDCA, SpaceX, Rocket Lab published data*  
*If this helped your project, an upvote is greatly appreciated! 🙏*
